# Compare Models

In this notebook I want to compare models - in particular I want to scrutinize, why almost all metrics of "non_aug_run" and "copy_author_run" are the same, except that in the latter one the Chamfer Distance is significantly smaller. This makes no sense to me, as Command and Argument Loss are almost the same.

Input: 
- Model 1
- Model 2
- sampel id

Output:
- printed list comapring MSE, Cmd.-Loss, Arg.-Loss, CD or Invalid
- Running scores

Then I can run the entire evaluation script and see in real time how it goes

In [1]:
import sys
import os
import random
import torch
import importlib
import csv
import numpy as np
sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
sys.path.append("..")
sys.path.append("../code")

from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE
from dataset import PointCloudEmbeddingSequenceDataset, PointCloudEmbeddingDataset
from models.DeepCAD.cadlib.macro import ALL_COMMANDS, CMD_ARGS_MASK, EOS_IDX, SOL_IDX, EXT_IDX, ARC_IDX
from models.DeepCAD.cadlib.visualize import vec2CADsolid, CADsolid2pc
from models.DeepCAD.utils import read_ply
from scipy.spatial import cKDTree as KDTree



In [2]:
import warnings

# Ignore any warning matching this pattern
warnings.filterwarnings("ignore", message=".* faces have been skipped due to null triangulation")


In [3]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

In [4]:
def load_pointnet(model_path):

    # Get torch model dir
    model_dir = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
    config = model_dir['config']
    print(f"Loading PointNet++ from {os.path.abspath(model_path)}:")
    for key, value in config.items():
        print(f"{key}: {value}")
    print("")

    # Create model
    model = importlib.import_module(config['model_type'])
    if 'architecture' in config:
        if config['architecture'] == 'own':
            classifier = model.get_model(256, normal_channel=False)
        elif config['architecture'] == "copy_author":
            classifier = model.get_model_copy_author(256, normal_channel=False)
        elif config['architecture'] == "tanh":
            classifier = model.get_model_tanh(256, normal_channel=False)
    else:
        classifier = model.get_model(256, normal_channel=False)
    criterion = model.get_loss_mse()
    classifier.apply(inplace_relu) 
    
    # Load pre-trained model
    state_dict = model_dir['model_state_dict']
    if 'module.' in next(iter(state_dict)):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    classifier.eval()
    classifier.load_state_dict(state_dict)
    
    return classifier, criterion

In [5]:
def load_deepcad(cfg):
    print("Loading DeepCAD: ")
    tr_agent = TrainerAE(cfg)
    tr_agent.load_ckpt(cfg.ckpt)
    tr_agent.net.eval()
    return tr_agent

In [6]:
def infer_pointnet(model, criterion, data):
    pc, z_target, vec_target, id = data['pc'], data['z'], data['tgt_vec'], data['id']
    pc = pc.unsqueeze(0)
    z_target = z_target.unsqueeze(0)
    with torch.no_grad():
        pc = pc.transpose(2, 1)
        z_pred, _ = model(pc)
        loss = criterion(z_pred, z_target)
    return z_pred, loss.item()

In [7]:
def infer_deepcad(model, z_pred, data):
    vec_target = data['tgt_vec']
    z_pred = z_pred.unsqueeze(dim = 1)
    with torch.no_grad():
        output = model.decode(z_pred)
        output["tgt_commands"] = vec_target[:, 0].unsqueeze(0)
        output["tgt_args"] = vec_target[:, 1:].unsqueeze(0)
        loss_dict = model.loss_func(output)
        cmd_loss = loss_dict['loss_cmd'].item()
        arg_loss = loss_dict['loss_args'].item()
        vec_pred = model.logits2vec(output)
        return vec_pred, cmd_loss, arg_loss

In [8]:
def calculate_ACC(results):

    TOLERANCE = 3

    # overall accuracy
    avg_cmd_acc = [] # ACC_cmd
    avg_param_acc = [] # ACC_param
    
    # accuracy w.r.t. each command type
    each_cmd_cnt = np.zeros((len(ALL_COMMANDS),))
    each_cmd_acc = np.zeros((len(ALL_COMMANDS),))

    # accuracy w.r.t each parameter
    args_mask = CMD_ARGS_MASK.astype(np.float32)
    N_ARGS = args_mask.shape[1]
    each_param_cnt = np.zeros([*args_mask.shape])
    each_param_acc = np.zeros([*args_mask.shape])

    B = results["tgt_commands"].shape[0]

    for i in range(B): 
        seq_length = list(results["tgt_commands"][i]).index(EOS_IDX)
        out_cmd = results["pred"][i,:seq_length,0]
        gt_cmd = results["tgt_commands"][i, :seq_length].numpy()
    
        out_param = results["pred"][i,:seq_length,1:]
        gt_param = results["tgt_args"][i, :seq_length].numpy()

        cmd_acc = (out_cmd == gt_cmd).astype(np.int32)
        param_acc = []
              
        for j in range(len(gt_cmd)):
            cmd = gt_cmd[j]
            each_cmd_cnt[cmd] += 1
            each_cmd_acc[cmd] += cmd_acc[j]
            if cmd in [SOL_IDX, EOS_IDX]:
                continue
        
            if out_cmd[j] == gt_cmd[j]: # NOTE: only account param acc for correct cmd
                tole_acc = (np.abs(out_param[j] - gt_param[j]) < TOLERANCE).astype(np.int32)
                if cmd == EXT_IDX:
                    tole_acc[-2:] = (out_param[j] == gt_param[j]).astype(np.int32)[-2:]
                elif cmd == ARC_IDX:
                    tole_acc[3] = (out_param[j] == gt_param[j]).astype(np.int32)[3]
                valid_param_acc = tole_acc[args_mask[cmd].astype(bool)].tolist()
                param_acc.extend(valid_param_acc)
                each_param_cnt[cmd, np.arange(N_ARGS)] += 1
                each_param_acc[cmd, np.arange(N_ARGS)] += tole_acc

        if len(param_acc) == 0: 
            param_acc = 0
        else:
            param_acc = np.mean(param_acc)
        
        avg_param_acc.append(param_acc)
        cmd_acc = np.mean(cmd_acc)
        avg_cmd_acc.append(cmd_acc)

    avg_cmd_acc = np.mean(avg_cmd_acc)
    avg_param_acc = np.mean(avg_param_acc)
    
    each_cmd_acc = each_cmd_acc / (each_cmd_cnt + 1e-6)

    # acc of each parameter type
    each_param_acc = each_param_acc * args_mask
    each_param_cnt = each_param_cnt * args_mask
    each_param_acc = each_param_acc / (each_param_cnt + 1e-6)
    return avg_cmd_acc, avg_param_acc

In [9]:
def calculate_CD(vec_pred, gt_pc_path, vec_target, data_id):

    seq_len = vec_target[:,0].tolist().index(EOS_IDX)
    vec_pred = vec_pred.squeeze()[:seq_len]

    try:
        shape = vec2CADsolid(vec_pred) # out vec only contains until target seq length
    except Exception as e:
        return float('nan') # Create CAD failed
    
    try:
        out_pc = CADsolid2pc(shape, 2000, data_id) # 2000 is the number of sampled points
    except Exception as e:
        return float('nan') # Create PC failed

    if np.max(np.abs(out_pc)) > 2: # normalize out-of-bound data
        out_pc = normalize_pc(out_pc)

    gt_pc = read_ply(gt_pc_path)
    sample_idx = random.sample(list(range(gt_pc.shape[0])), 2000)
    gt_pc = gt_pc[sample_idx]

    cd = chamfer_dist(gt_pc, out_pc)
    return cd


In [10]:
def chamfer_dist(gt_points, gen_points, offset=0, scale=1):
    gen_points = gen_points / scale - offset

    # one direction
    gen_points_kd_tree = KDTree(gen_points)
    one_distances, one_vertex_ids = gen_points_kd_tree.query(gt_points)
    gt_to_gen_chamfer = np.mean(np.square(one_distances))

    # other direction
    gt_points_kd_tree = KDTree(gt_points)
    two_distances, two_vertex_ids = gt_points_kd_tree.query(gen_points)
    gen_to_gt_chamfer = np.mean(np.square(two_distances))

    return gt_to_gen_chamfer + gen_to_gt_chamfer

In [11]:
def normalize_pc(points):
    scale = np.max(np.abs(points))
    points = points / scale
    return points

In [12]:
def print_header():
    print("MSE * 10^-3, ACC * 10^-2, CD * 10^-3")
    print(f"{'i':<4} | {'id':<8} | {'mse1':<6} | {'mse2':<6} | {'cmd_loss1':<10} | {'cmd_loss2':<10} | {'arg_loss1':<10} | {'arg_loss2':<10} | {'cmd_acc1':<8} | {'arg_acc1':<8} | {'cmd_acc2':<8} | {'arg_acc2':<8} | {'cd1':<8} | {'cd2':<8} | {'rs_cd1':<8} | {'rs_cd2':<8}")
    print("-" * 175)


def print_line(data):
    print(
        f"{data['i']:<4} | {data['id']:<8} | {data['mse1']*1e3:6.2f} | {data['mse2']*1e3:6.2f} | "
        f"{data['cmd_loss1']:10.4f} | {data['cmd_loss2']:10.4f} | {data['arg_loss1']:10.4f} | {data['arg_loss2']:10.4f} | "
        f"{data['cmd_acc1']*1e2:8.2f} | {data['arg_acc1']*1e2:8.2f} | {data['cmd_acc2']*1e2:8.2f} | {data['arg_acc2']*1e2:8.2f} | "
        f"{data['cd1']*1e3:8.2f} | {data['cd2']*1e3:8.2f} | {data['rs_cd1']*1e3:8.2f} | {data['rs_cd2']*1e3:8.2f} | "
    )


## Start

In [13]:
pointnet1_path = "../models/trained_models/fifth_official_run/best.pth"
pointnet2_path = "../models/trained_models/copy_author_step_adv_run/best.pth"
cfg = ConfigAE('test', model_path="../data/latent", parse=False)

In [14]:
pointnet1, criterion1 = load_pointnet(pointnet1_path)
pointnet2, criterion2 = load_pointnet(pointnet2_path)
deepcad = load_deepcad(cfg)

Loading PointNet++ from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/models/trained_models/fifth_official_run/best.pth:
learning_rate: 0.001
batch_size: 192
max_epochs: 500
optimizer: Adam
model_type: pointnet2_cls_ssg
save_interval: 30
early_stopping: 500
start_time: 2025-01-31_08-47-16
lr_type: cosine
final_epoch: 494
training_time_min: 6384.63

Loading PointNet++ from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/models/trained_models/copy_author_step_adv_run/best.pth:
learning_rate: 0.0001
batch_size: 384
max_epochs: 500
optimizer: Adam
model_type: pointnet2_cls_msg
save_interval: 20
early_stopping: 50
start_time: 2025-03-19_19-35-05
lr_type: step_adv
msg: True
augmentation: False
gpu: True
architecture: copy_author
final_epoch: 351
training_time_min: 2645.26

Loading DeepCAD: 
Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model

In [15]:
start = True
csv_name = "test.csv"
if start:
    dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')
    fieldnames = [
        'i', 'id', 'mse1', 'mse2', 
        'cmd_loss1', 'cmd_loss2', 'arg_loss1', 'arg_loss2',
        'cmd_acc1', 'arg_acc1', 'cmd_acc2', 'arg_acc2', 
        'cd1', 'cd2', 'rs_cd1', 'rs_cd2'
    ]
    col_width = print_header()
    with open(os.path.join('experiments', csv_name), 'w', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
    
    for i, data in enumerate(dataset):
        # Infer PointNets
        z_pred1, mse1 = infer_pointnet(pointnet1, criterion1, data)
        z_pred2, mse2 = infer_pointnet(pointnet2, criterion2, data)
    
        # Infer DeepCAD
        vec_pred1, cmd_loss1, arg_loss1 = infer_deepcad(deepcad, z_pred1, data)
        vec_pred2, cmd_loss2, arg_loss2 = infer_deepcad(deepcad, z_pred2, data)
    
        # Calculate Accuracy
        cmd_acc1, arg_acc1 = calculate_ACC({"tgt_commands": data['tgt_vec'].unsqueeze(0)[:,:,0], "tgt_args": data['tgt_vec'].unsqueeze(0)[:,:,1:], "pred": vec_pred1})
        cmd_acc2, arg_acc2 = calculate_ACC({"tgt_commands": data['tgt_vec'].unsqueeze(0)[:,:,0], "tgt_args": data['tgt_vec'].unsqueeze(0)[:,:,1:], "pred": vec_pred2})

        print(vec_pred1.shape, type(vec_pred1))
        print(data['tgt_vec'].shape, type(data['tgt_vec']))
        cd1 = calculate_CD(vec_pred1, dataset.get_pc_path(i), data['tgt_vec'], data['id'])
        cd2 = calculate_CD(vec_pred2, dataset.get_pc_path(i), data['tgt_vec'], data['id'])
    
        existing_cd1 = []
        existing_cd2 = []
        with open(os.path.join('experiments', csv_name), 'r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                if row['cd1'] != 'nan': 
                    existing_cd1.append(float(row['cd1']))
                if row['cd2'] != 'nan':
                    existing_cd2.append(float(row['cd2']))
        
        # Compute running medians (handle NaNs)
        rs_cd1 = np.nanmedian(existing_cd1 + [cd1])
        rs_cd2 = np.nanmedian(existing_cd2 + [cd2])
        metrics_dict = {
                'i': i,
                'id': data['id'],
                'mse1': mse1,
                'mse2': mse2,
                'cmd_loss1': cmd_loss1,
                'cmd_loss2': cmd_loss2,
                'arg_loss1': arg_loss1,
                'arg_loss2': arg_loss2,
                'cmd_acc1': cmd_acc1,
                'arg_acc1': arg_acc1,
                'cmd_acc2': cmd_acc2,
                'arg_acc2': arg_acc2,
                'cd1': cd1,
                'cd2': cd2,
                'rs_cd1': rs_cd1,
                'rs_cd2': rs_cd2
            }
        with open(os.path.join('experiments', csv_name), 'a', newline='') as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writerow(metrics_dict)
            
        print_line(metrics_dict)
        break





MSE * 10^-3, ACC * 10^-2, CD * 10^-3
i    | id       | mse1   | mse2   | cmd_loss1  | cmd_loss2  | arg_loss1  | arg_loss2  | cmd_acc1 | arg_acc1 | cmd_acc2 | arg_acc2 | cd1      | cd2      | rs_cd1   | rs_cd2  
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
(1, 60, 17) <class 'numpy.ndarray'>
torch.Size([60, 17]) <class 'torch.Tensor'>
0    | 00250456 |   9.24 |   9.45 |     0.0000 |     0.0000 |     0.0000 |     0.0001 |   100.00 |   100.00 |   100.00 |   100.00 |     4.21 |     4.22 |     4.21 |     4.22 | 


## Show the results

In [48]:
import pandas as pd
df = pd.read_csv('experiments/nonaug_vs_copyauthor.csv')
id = df['id']

In [49]:
def create_comparison_table(row):
    metrics = ["mse", "cmd_loss", "arg_loss", "cmd_acc", "arg_acc", "cd", "rs_cd"]
    model1_values = [round(row[f"{metric}1"].values[0], 5) for metric in metrics]
    model2_values = [round(row[f"{metric}2"].values[0], 5) for metric in metrics]
    
    comparison_df = pd.DataFrame({
        "Metric": metrics,
        "Model 1": model1_values,
        "Model 2": model2_values
    })

    return comparison_df
    
comparison_id = 716729
row = df.loc[df['id'] == comparison_id]
comparison_table = create_comparison_table(row)
comparison_table

,Metric,Model 1,Model 2
0,mse,0.11104,0.10226
1,cmd_loss,2.11552,2.72374
2,arg_loss,11.31137,13.19744
3,cmd_acc,0.66667,0.66667
4,arg_acc,0.53333,0.53333
5,cd,2.38854,0.25674
6,rs_cd,0.02011,0.01409


In [50]:
avg_acc_cmd1 = round(np.mean(df['cmd_acc1']) * 1e2, 2)
avg_acc_arg1 = round(np.mean(df['arg_acc1']) * 1e2, 2)
median_cd1 = round(np.nanmedian(df['cd1']) * 1e3, 2)
ir1 = round(np.isnan(np.array(df['cd1'])).sum()/len(df['cd1']) * 1e2, 2)
avg_mse1 = round(np.mean(df["mse1"]) * 1e3, 2)
avg_cmd_loss1 = round(np.mean(df["cmd_loss1"]), 2)
avg_arg_loss1 = round(np.mean(df["arg_loss1"]), 2)

metrics_model1 = [avg_acc_cmd1, avg_acc_arg1, median_cd1, ir1, avg_mse1, avg_cmd_loss1, avg_arg_loss1]

In [51]:
avg_acc_cmd2 = round(np.mean(df['cmd_acc2']) * 1e2, 2)
avg_acc_arg2 = round(np.mean(df['arg_acc2']) * 1e2, 2)
median_cd2 = round(np.nanmedian(df['cd2']) * 1e3, 2)
ir2 = round(np.isnan(np.array(df['cd2'])).sum()/len(df['cd2']) * 1e2, 2)
avg_mse2 = round(np.mean(df["mse2"]) * 1e3, 2)
avg_cmd_loss2 = round(np.mean(df["cmd_loss2"]), 2)
avg_arg_loss2 = round(np.mean(df["arg_loss2"]), 2)

metrics_model2 = [avg_acc_cmd2, avg_acc_arg2, median_cd2, ir2, avg_mse2, avg_cmd_loss2, avg_arg_loss2]

In [52]:
df_summary = pd.DataFrame(columns=["acc_cmd", "acc_args", "median_cd", "IR", "avg. MSE", "avg. cmd_loss", "avg. arg_loss"])
df_summary.loc[len(df_summary)] = metrics_model1
df_summary.loc[len(df_summary)] = metrics_model2

In [53]:
df_summary

,acc_cmd,acc_args,median_cd,IR,avg. MSE,avg. cmd_loss,avg. arg_loss
0,75.00,66.61,20.24,17.94,72.63,6.24,10.36
1,74.89,66.81,14.00,17.53,71.78,6.14,10.79


## Evaluation

I want to see, where the largest Chamfer Distance differences come from. A smaller CD is better, therefore we substract CD2 (better model) from CD1 (worse model) and look at the largest positive differences.

In [76]:
cd_diff = round((df['cd1'] - df['cd2']) * 1e3, 2)
cmd_loss_diff = round(df['cmd_loss1'] - df['cmd_loss2'], 2)
arg_loss_diff = round(df['arg_loss1'] - df['arg_loss2'], 2)
cmd_acc_diff = round((df['cmd_acc1'] - df['cmd_acc2']) * 1e2, 2)
arg_acc_diff = round((df['arg_acc1'] - df['arg_acc2'])* 1e2, 2)
mse_diff = round((df['mse1'] - df['mse2']) * 1e3, 2)

df_compare = pd.DataFrame({'id': id, 
                           'cd': cd_diff, 
                           'cmd_loss': cmd_loss_diff, 
                           'arg_loss': arg_loss_diff, 
                           'cmd_acc': cmd_acc_diff, 
                           'arg_acc': arg_acc_diff,
                           'mse': mse_diff})
df_compare = df_compare.sort_values(by='cd', ascending=False).reset_index(drop=True)




In [77]:
df_compare

,id,cd,cmd_loss,arg_loss,cmd_acc,arg_acc,mse
0,716729,2131.80,-0.61,-1.89,0.00,0.00,8.78
1,138168,1736.42,-1.64,-2.58,-5.00,0.00,3.91
2,465682,1570.68,-0.00,7.62,0.00,10.53,5.82
3,608830,1569.20,0.00,3.53,0.00,0.00,4.28
4,208627,1549.28,-6.83,-0.17,6.67,1.67,-6.04
...,...,...,...,...,...,...,...
8033,96611,NaN,0.56,-5.57,0.00,2.27,-4.22
8034,386810,NaN,3.73,11.95,-22.22,-64.00,6.34
8035,839500,NaN,5.42,0.98,-6.67,0.00,1.40
8036,137527,NaN,0.70,0.01,-6.25,9.82,0.77


In [34]:
import pandas as pd

# Example data
ids = [1, 2, 3]
diffs = [0.2, 0.9, 0.8]

# Create the DataFrame
df = pd.DataFrame({'id': ids, 'diff': diffs})

# Sort by 'diff' in descending order
df_sorted = df.sort_values(by='diff', ascending=False).reset_index(drop=True)

# Extract the sorted id list
sorted_ids = df_sorted['id'].tolist()

print("Sorted IDs:", sorted_ids)


Sorted IDs: [2, 3, 1]


## Next

- see where the largest CD differences are
- for these samples look if arg and cad loss

Erkentniss:
- Avg loss für cmd und arg wird in pc2cad falsch berechnet. ich nehme das average von den batches (die jeweils das average beinhalten) - die samples im letzten batch haben dadurch verhältnismäßig viel gewicht, da das letzte batch nicht ganz voll ist
    - Lösung: In der loss funktion die losses addieren und nicht den default mean nehmen, und dann durch die Anzahl der samples teilen
    - Andererseits ist es glaube nicht sooo wichtig gerade, denn die anderen metrics sind ja korrekt, und der Loss ist nicht so aussagekräftig
 
Done
- create overview of model results

Created Files:
- model1_vs_model2.csv